# Phase 1.1 - Document Forgery Detection

Fine-tunes a ResNet-50 to classify a document image as **authentic** or **tampered**, using MIDV-2020 (Roboflow Universe, CC BY 4.0) as authentic samples and programmatically generated synthetic tampering (copy-paste, splice, photo-swap) as the tampered class.

Colab-compatible: run the setup cell first. Locally, just make sure you're running this from the repo root with `venv` active.

## 0. Setup

On Colab this clones the repo (replace `REPO_URL` with your own GitHub URL after pushing) and installs dependencies. Locally, it just makes sure `src/` is importable and skips straight through.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/YOUR_USERNAME/ai-identity-shield.git"  # <- replace after pushing

if IN_COLAB:
    if not Path("ai-identity-shield").exists():
        !git clone {REPO_URL}
    %cd ai-identity-shield
    !pip install -q -r requirements.txt
    REPO_ROOT = Path.cwd()
else:
    # Find the repo root by walking up from the notebook's cwd until we see requirements.txt.
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)


## 1. Configure secrets

Needs a free Roboflow API key (https://app.roboflow.com/settings/api) to download MIDV-2020. Locally this is read from `.env` (copy `.env.example` -> `.env`). On Colab, either use `google.colab.userdata` (Secrets panel) or paste it into the prompt below - it is never written to a file in the repo.

In [ ]:
import os

if "ROBOFLOW_API_KEY" not in os.environ:
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
        except Exception:
            import getpass
            os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")
    else:
        from dotenv import load_dotenv
        load_dotenv()

assert os.environ.get("ROBOFLOW_API_KEY"), "ROBOFLOW_API_KEY not set - see .env.example"

## 2. Download MIDV-2020 and generate synthetic tampering

`data/` and `models/` are gitignored - these scripts are idempotent, safe to re-run.

In [ ]:
from scripts.download_midv2020 import main as download_midv2020
from scripts.generate_synthetic_tampering import main as generate_tampering

download_midv2020()
generate_tampering()

## 3. Inspect a sample authentic/tampered pair

In [ ]:
import csv

import matplotlib.pyplot as plt
from PIL import Image

from src.common.config import MIDV2020_TAMPER_DIR, ROOT

with open(MIDV2020_TAMPER_DIR / "train_manifest.csv", newline="") as f:
    rows = list(csv.DictReader(f))

authentic_row = next(r for r in rows if r["label"] == "0")
tampered_row = next(r for r in rows if r["source_image"] == authentic_row["source_image"] and r["label"] == "1")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(ROOT / authentic_row["filepath"]))
axes[0].set_title("authentic")
axes[1].imshow(Image.open(ROOT / tampered_row["filepath"]))
axes[1].set_title(f"tampered ({tampered_row['technique']})")
for ax in axes:
    ax.axis("off")
plt.show()

## 4. Train the ResNet-50 forgery detector

Saves the best checkpoint (by validation F1) to `models/forgery_resnet50.pt`. Runs fine on a free Colab T4, or on a local GPU with a few GB of VRAM - drop `epochs`/`batch_size` if you're on CPU only.

In [ ]:
from src.forgery_detection.train import train

train(epochs=8, batch_size=16, lr=1e-4)

## 5. Evaluate: accuracy / precision / recall / F1 / confusion matrix

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.common.config import MODELS_DIR
from src.forgery_detection.dataset import ForgeryDataset
from src.forgery_detection.evaluate import evaluate, save_confusion_matrix
from src.forgery_detection.model import load_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(MODELS_DIR / "forgery_resnet50.pt", device)
valid_loader = DataLoader(
    ForgeryDataset(MIDV2020_TAMPER_DIR / "valid_manifest.csv", train=False), batch_size=16
)

metrics = evaluate(model, valid_loader, device)
print({k: v for k, v in metrics.items() if k != "confusion_matrix"})
save_confusion_matrix(metrics["confusion_matrix"], "confusion_matrix.png")
plt.imshow(Image.open("confusion_matrix.png"))
plt.axis("off")
plt.show()

## 6. Grad-CAM: what is the model looking at?

Highlights the image region that drove each prediction - for tampered images this should land on the spliced/copy-moved/swapped region.

In [ ]:
from src.common.config import OUTPUTS_DIR
from src.forgery_detection.gradcam import run_gradcam

gradcam_dir = OUTPUTS_DIR / "gradcam"
run_gradcam(MODELS_DIR / "forgery_resnet50.pt", split="valid", n_samples=6, out_dir=gradcam_dir)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, path in zip(axes.flat, sorted(gradcam_dir.glob("*.png"))):
    ax.imshow(Image.open(path))
    ax.set_title(path.stem, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. OCR field extraction demo (TrOCR on MIDV-2020's annotated field boxes)

MIDV-2020's own annotations give us ground-truth box locations for `primary_identifier` (name), `date_of_birth`, and `document_number` - this reuses them directly, as suggested in the project's dataset plan, instead of needing a separate OCR dataset.

In [ ]:
from src.common.coco_utils import load_coco_split
from src.common.config import MIDV2020_DIR
from src.ocr.extract import TrOCRFieldExtractor
from src.ocr.validate import validate_fields

records = load_coco_split(MIDV2020_DIR / "valid")
sample = records[0]
sample_image = Image.open(MIDV2020_DIR / "valid" / sample.file_name).convert("RGB")

trocr = TrOCRFieldExtractor(device=device)
raw_fields = trocr.extract_fields_from_boxes(sample_image, sample.fields)
print("raw OCR output:", raw_fields)
print("validated:", validate_fields(raw_fields))

## Next steps

- Phase 1 continues with the FastAPI backend (`backend/app/main.py`) and React frontend (`frontend/`), which serve this trained checkpoint plus the Donut-based zero-shot OCR extractor (`src/ocr/extract.py::DonutFieldExtractor`) for arbitrary uploads with no known field boxes.
- Phase 2 adds face matching, deepfake detection, and the combined trust-score verification engine - see the repo README for status.